# CSCI/MATH 485 Assignment
## Customer Churn Prediction with XGBoost
## Starter Notebook

This notebook is compatible with **Jupyter Notebook** and **Google Colab**.

This starter code is only to get you started. You can change any of the existing code here to complete all the tasks.

Complete all `TODO` sections. Make sure your final submission includes:
- data analysis,
- a tuned XGBoost model,
- your chosen main evaluation metric and justification,
- interpretation of top important features,
- and a final comparison with the baseline model.


## 1. Setup


In [1]:
# If you are using Google Colab, uncomment the next line if xgboost is not installed.
# !pip install xgboost


In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier


## 2. Load the Dataset

**Instructions for students**
- Load the IBM Telco Customer Churn dataset.
- Display the first few rows.
- Confirm the dataset shape.
- If the url doesn't work for you, download the csv file from the Canvas Assignment#4

In [36]:
import csv


url = "https://raw.githubusercontent.com/plotly/datasets/master/telco-customer-churn-by-IBM.csv"
df = pd.read_csv(url)

# Display the first 5 rows of the dataset (done below)
df.head()

# TODO:
print("Shape of the dataset:", df.shape)
print(df.head())


Shape of the dataset: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Str

## 3. Data Exploration

Complete the following:
- Print all column names
- Show data types
- Count missing values in each column
- Show the distribution of the target variable
- Write a short note: Is this a classification or regression problem? Why is this useful in business?


In [13]:
print("Columns:")
print(df.columns.tolist())


print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df['Churn'].value_counts())


# Extra: display other information of the dataset that you think can be useful




Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Missing values:
customerID          0
gender       

**TODO (write your answer below):**

1. What kind of machine learning problem is this?
2. Why is churn prediction important in a business setting?


Answer:
1. This is a binary classification problem, where we're predicting churn/no churn. 
2. Churn is important in a business setting as customer churn directly impacts business profits and place in the market. If churn rates are high, consumers are leaving the business for competitiors, possibly due to facotrs such as poor service, false marketing, low quality products, etc. whereas low churn rates give insight to business and product strengths, uniqueness, and ideal organization. 


## 4. Basic Cleaning

Complet the following:
- Identify whether there is an ID column that should be removed
- Convert the target column into binary form if needed
- Convert any numeric-looking columns stored as text into numeric values


In [14]:
# Make a copy so the original data remains unchanged
df_clean = df.copy()

df_clean = df_clean.drop(columns=['customerID'])
df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

df_clean.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 5. Define Features and Target

Complet the following:
- Define `X` and `y`
- Set the correct target column


In [15]:
target_col = "Churn"

X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


Feature matrix shape: (7043, 19)
Target shape: (7043,)


## 6. Identify Numeric and Categorical Features

Complet the following:
- Create a list of numeric columns
- Create a list of categorical columns


In [16]:
numeric_features = X.select_dtypes(include=["float64"]).columns.tolist()

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

binary_features = [
    col for col in X.select_dtypes(include=["int64"]).columns
    if X[col].nunique() == 2
]
categorical_features += binary_features

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


Numeric features:
['MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']


## 7. Train/Test Split

Complet the following:
- Split the dataset into training and testing sets
- Use stratification if appropriate


In [17]:
# TODO:
# Create train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (5634, 19)
X_test shape: (1409, 19)


## 8. Preprocessing Pipelines

Build preprocessing for:
- numeric features
- categorical features

Then combine them into a `ColumnTransformer`.


In [18]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 9. Baseline Model: Logistic Regression

Complet the following:
- Train a Logistic Regression model as the baseline
- Generate predictions
- Evaluate using multiple metrics
- You may need to adjust `max_iter` if the model is not converging.


In [19]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=3000))
])

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)


In [20]:
baseline_preds = baseline_model.predict(X_test)
baseline_probs = baseline_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, baseline_preds))
print("Precision:", precision_score(y_test, baseline_preds))
print("Recall:", recall_score(y_test, baseline_preds))
print("F1-score:", f1_score(y_test, baseline_preds))
print("ROC-AUC:", roc_auc_score(y_test, baseline_probs))

Accuracy: 0.7934705464868701
Precision: 0.6325878594249201
Recall: 0.5294117647058824
F1-score: 0.5764192139737991
ROC-AUC: 0.8298251052726757


In [21]:
print(classification_report(y_test, baseline_preds))
print(confusion_matrix(y_test, baseline_preds))


              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.63      0.53      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409

[[920 115]
 [176 198]]


## 10. Choose Your Main Evaluation Metric

Choose **one main metric** for this churn problem.

You must explain:
- which metric you chose,
- why it is appropriate,
- and why it is more informative than accuracy alone for this problem.


I selected the metric `ROC-AUC` because it measures the distinguishes between churn / no churn with great accuracy, which gives us a better understanding of the discrepency between true positives and false positives. This dataset is imbalanced, as simply observed in the confusion matrix. 

## 11. XGBoost Model

Complet the following:
- Build an XGBoost pipeline
- Tune at least 3 hyperparameters
- Use either `GridSearchCV` or your own tuning approach


In [22]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        eval_metric="auc",
        random_state=42
    ))
])

param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8, 1.0]
}

param_grid


{'classifier__n_estimators': [50, 100],
 'classifier__max_depth': [3, 5],
 'classifier__learning_rate': [0.05, 0.1],
 'classifier__subsample': [0.8, 1.0]}

In [23]:
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring="roc_auc", 
    cv=3,
    n_jobs=-1,
    verbose=1
)


grid_search.fit(X_train, y_train)


Fitting 3 folds for each of 16 candidates, totalling 48 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'classifier__learning_rate': [0.05, 0.1], 'classifier__max_depth': [3, 5], 'classifier__n_estimators': [50, 100], 'classifier__subsample': [0.8, 1.0]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [24]:
print("Best parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_

best_model

Best parameters: {'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 100, 'classifier__subsample': 0.8}


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 12. Evaluate the Tuned XGBoost Model

Evaluate XGBoost using the same metrics as the baseline.


In [25]:
xgb_preds = best_model.predict(X_test)
xgb_probs = best_model.predict_proba(X_test)[:, 1]


In [26]:
print("Accuracy:", accuracy_score(y_test, xgb_preds))
print("Precision:", precision_score(y_test, xgb_preds))
print("Recall:", recall_score(y_test, xgb_preds))
print("F1-score:", f1_score(y_test, xgb_preds))
print("ROC-AUC:", roc_auc_score(y_test, xgb_probs))

Accuracy: 0.8062455642299503
Precision: 0.6771929824561403
Recall: 0.516042780748663
F1-score: 0.5857359635811836
ROC-AUC: 0.8471389082642279


In [27]:
print(classification_report(y_test, xgb_preds))
print(confusion_matrix(y_test, xgb_preds))


              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.68      0.52      0.59       374

    accuracy                           0.81      1409
   macro avg       0.76      0.71      0.73      1409
weighted avg       0.80      0.81      0.80      1409

[[943  92]
 [181 193]]


## 13. Feature Importance

Use the trained XGBoost model to:
- extract feature importances,
- recover transformed feature names,
- display the top 5 to 10 most important features.


In [28]:
fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_xgb = best_model.named_steps["classifier"]


In [29]:
feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_xgb.feature_importances_
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [31]:
df_feature_importances = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

df_feature_importances.head(10)

,feature,importance
34,cat__Contract_Month-to-month,0.389132
14,cat__InternetService_Fiber optic,0.105695
16,cat__OnlineSecurity_No,0.082815
25,cat__TechSupport_No,0.072400
41,cat__PaymentMethod_Electronic check,0.038295
1,num__TotalCharges,0.027212
33,cat__StreamingMovies_Yes,0.024960
36,cat__Contract_Two year,0.022946
37,cat__PaperlessBilling_No,0.022738
13,cat__InternetService_DSL,0.022316


## 14. Interpret the Top Features

Write a short interpretation of the most important features.

Your explanation should:
- use plain language,
- connect features to churn behavior,
- and explain what the company might learn from them.


These most important features give the business predictive insight into what is impact churn rates, whether churn or no turn. These factors, such as whether or not customers have online security, use tech support, or their contract is month-to-month help predict churns. The company can learn which areas of customer experience and services might need better marketing, restructuring, new strategy, or is contributing as they hoped. 


## 15. Final Comparison: Logistic Regression vs XGBoost

Compare the two models using your results.

Your discussion should answer:
- Which model performed better?
- On which metric(s)?
- Why might XGBoost perform better on this dataset?
- What is one limitation of XGBoost?


In [35]:


df_metrics = pd.DataFrame({
    "Baseline Metrics": [
        accuracy_score(y_test, baseline_preds),
        precision_score(y_test, baseline_preds),
        recall_score(y_test, baseline_preds),
        f1_score(y_test, baseline_preds),
        roc_auc_score(y_test, baseline_probs)
    ],  
    "XGBoost Metrics": [
        accuracy_score(y_test, xgb_preds),
        precision_score(y_test, xgb_preds),
        recall_score(y_test, xgb_preds),
        f1_score(y_test, xgb_preds),
        roc_auc_score(y_test, xgb_probs)
    ]
})

df_metrics.index = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]

df_metrics

,Baseline Metrics,XGBoost Metrics
Accuracy,0.793471,0.806246
Precision,0.632588,0.677193
Recall,0.529412,0.516043
F1-score,0.576419,0.585736
ROC-AUC,0.829825,0.847139


The XGBoost model outperformed the Logistic Regression model across all main evaluation metrics, including accuracy, precision, F1-score, and ROC-AUC, except for recall due to the slightly increased number of false negatives. This may be because XGBoost is more particular about positives and/or because I selected `ROC-AUC` as my scoring metric. XGBoost performs better on this dataset because it is imbalanced and XGBoost handles imbalanced datasets and missing values better due to the precise decision boundaries and enhanced handling.
One limitation of XGBoost is the difficulty in interpreting results, as they are not as direct. Additionally, XGBoost requires more perameters than Logistic Regression, which further adds to the complexity of the model.


## 16. Suggested Submission Checklist

Before submitting, make sure your notebook includes:
- completed code cells,
- outputs for each section,
- your selected main evaluation metric and justification,
- tuned XGBoost hyperparameters,
- feature importance results,
- interpretation of important features,
- and a final comparison of the two models.
